# Networks

A result file describes a network as locations with the names the model gave
them: a node id, or a reach and a distance along it. `mikeio1d.network` turns
that into a graph whose nodes are flat integers, carrying the timeseries each
location holds, and translates between the two namings.

The module needs `networkx` and `xarray`:

```
pip install mikeio1d[network]
```

It is provisional: the shape of what it returns may still change.


In [ ]:
import matplotlib.pyplot as plt
import networkx as nx

from mikeio1d.network import Network

network = Network.open("../tests/testdata/network.res1d")
network

## The graph

One reach becomes a chain of edges: its start node, its breakpoints in order,
its end node. Every edge carries the distance between its two ends, where that
is known.

An edge is marked `boundary` where its two ends are the same place. A reach's
own end gridpoint is promoted to a breakpoint, which then sits exactly on the
node the reach starts or ends at, so the edge joining them has zero length and
exists only to say the two are one location.

In [ ]:
graph = network.graph

boundary = [(u, v) for u, v, data in graph.edges(data=True) if data["boundary"]]
interior = [(u, v) for u, v, data in graph.edges(data=True) if not data["boundary"]]
print(f"{graph.number_of_nodes()} nodes, {len(interior)} interior edges, {len(boundary)} boundary edges")

layout = nx.spring_layout(graph, seed=1)
fig, ax = plt.subplots(figsize=(9, 7))
nx.draw_networkx_nodes(graph, layout, node_size=8, ax=ax)
nx.draw_networkx_edges(graph, layout, edgelist=interior, width=1, ax=ax)
nx.draw_networkx_edges(graph, layout, edgelist=boundary, width=2, edge_color="tab:orange", ax=ax)
ax.set_axis_off()

## The data

Every location's timeseries, as a dataframe with the node id and the quantity
in the columns.

In [ ]:
network.to_dataframe()

As a dataset, each node also carries the name it had before it became an
integer: `name` for a node, `reach` and `distance` for a breakpoint. The empty
half says which of the two it is, so nothing that reads the dataset has to keep
the network around to interpret it.

In [ ]:
ds = network.to_dataset()
ds

## Finding a location

`find()` goes from the model's names to the integer the graph uses.

In [ ]:
network.find(node=["100", "98"])

A breakpoint is named by its reach and a distance along it. `"start"` and
`"end"` reach the reach's own nodes.

In [ ]:
print(network.find(reach="100l1", distance=23.841))
print(network.find(reach="100l1", distance="start"))

`recall()` is the reverse: it gives back the names a node id came from.

In [ ]:
network.recall([34, 252])

## EPANET, and the files beside it

An EPANET run writes up to three files worth reading. The `.res` holds the
network and its main results; a `.resx` holds extra results for the same
network; and the `.inp` input file is the only one of the three carrying reach
lengths. Left alone, `open` picks up whichever of them share the result file's
folder and stem.

Pass `companions=[]` to read the result on its own, or name the files to read
exactly those.

In [ ]:
epanet = Network.open("../tests/testdata/epanet.res")
alone = Network.open("../tests/testdata/epanet.res", companions=[])

print("with companions:", epanet._reaches["10"].length, "|", "Volume" in epanet.quantities)
print("on its own:     ", alone._reaches["10"].length, "|", "Volume" in alone.quantities)

Without the `.inp`, no reach has a length, and the graph says so: the edges
whose length is unknown carry `None` rather than a guess.

In [ ]:
unknown = sum(data["length"] is None for *_, data in alone.graph.edges(data=True))
print(f"{unknown} of {alone.graph.number_of_edges()} edges have no length without the .inp")